In [33]:
# !pip install koreanize_matplotlib

import warnings
import koreanize_matplotlib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as  plt
from matplotlib_venn import venn2
import plotly.express as px
import plotly.graph_objects as go
import ast


# 그래프 해상도 높이기
try:
    %config InlineBackend.figure_format = 'retina'
except Exception as e:
    print(f'💩 {e}')



# 경고 무시
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)  # 출력할 너비를 넉넉하게 조정
pd.set_option('display.expand_frame_repr', False)  # 옆으로 길어져도 줄바꿈 없이 출력
pd.set_option('display.max_colwidth', None)  # 긴 문자열도 생략 없이 출력

try:
    from google.colab import drive
    drive.mount('/content/drive')

    import os
    os.chdir('/content/drive/MyDrive/파트4')
    print('✅ Succesful access google_drive_directory')
    
except Exception as e:
    print('🤗 Hello vscode')
        
## get_df 함수
def get_df(db_name, table_name):
    if table_name in ['accounts_user', 'accounts_blockrecord']:
        table_name = pd.read_parquet(
            f"gs://high_project/{db_name}/{table_name}.parquet", 
            storage_options={'token' : API_KEY_PATH})
    else:
        table_name = pd.read_csv(
            f"gs://high_project/{db_name}/{table_name}.csv",
            storage_options={'token' : API_KEY_PATH}
            )
    return table_name
    

## literal_eval 형변환 함수
def to_literal_eval(df, column):
    return  df[column].apply(lambda x: ast.literal_eval(x) if x != '[]' else [])


# 리스트 내에 드랍 유저가 있는지 확인하는 함수
def find_drop_users(df, column):
    drop_users = [831956, 1580627, 1580689, 1580626, 995177]
    print(f'{column}:')
    for i in drop_users:
        count_drop_rows = len(df[df[column].apply(lambda x: i in x)])
        if count_drop_rows != 0:
            print(f"‼️ 관리자 {i}가 포함된 행 {count_drop_rows}개 존재")
        else:
            print(f"✅ 관리자 {i} 포함행 없음")
            
            
# 데이트타임형으로 변환 및 기간 전처리
def get_datetime(df, column):
    try:
        df[column] =  pd.to_datetime(df[column])
        print(f'✅ {column} 데이트 타입 형변환 완')
    except Exception as e:
        print(f'💩 {e}')

    return df[df[column] < '2023-09-01']

    
 
# 데이트 기간 출력 및 교집합 설정   
def get_date_range(df1, col1, df1_name, df2, col2, df2_name):
    print(f"{df1_name}: {df1[col1].min()} ~ {df1[col1].max()}")
    print(f"{df2_name}: {df2[col2].min()} ~ {df2[col2].max()}")
    start_date = max(df1[col1].min(), df2[col2].min())
    end_date = min(df1[col1].max(), df2[col2].max())  
    print(f"두 시간 범위의 교집합: {start_date} ~ {end_date}")
    
    answer = input('교집합 기간으로 설정하고 싶으신가요? (Y/N): ')
    if answer.upper() == 'Y':
        df1_filtered = df1[(df1[col1] >= start_date) & (df1[col1] <= end_date)].copy()
        df2_filtered = df2[(df2[col2] >= start_date) & (df2[col2] <= end_date)].copy()
        print("✅ 교집합 기간으로 필터링 완료되었습니다.")
        
        return df1_filtered, df2_filtered
    else:
        print("필터링 없이 원래 데이터프레임을 반환합니다.")
        return df1, df2
    

            
API_KEY_PATH ='/home/project_yujin/API_KEY/sprintda03-yujin.json'

🤗 Hello vscode


## accounts_pointhistory

In [29]:
'''데이터 불러오기 및 컬럼순서 별경'''
accounts_pointhistory = get_df('votes', 'accounts_pointhistory')
accounts_pointhistory = accounts_pointhistory[['id', 'user_id', 'user_question_record_id', 'delta_point', 'created_at']]

'''중복값 재거'''
accounts_pointhistory = accounts_pointhistory[~accounts_pointhistory.loc[:, [ 'user_id', 'user_question_record_id', 'delta_point', 'created_at']].duplicated()]
# accounts_pointhistory = accounts_pointhistory.dropna()

'''8월까지 날짜 필터링'''
# accounts_pointhistory['user_question_record_id'] = accounts_pointhistory['user_question_record_id'].astype(int)
accounts_pointhistory = get_datetime(accounts_pointhistory, 'created_at')
accounts_pointhistory.head()

✅ created_at 데이트 타입 형변환 완


,id,user_id,user_question_record_id,delta_point,created_at
0,790629,849436,771777.0,9,2023-04-28 12:27:49
1,790652,849436,771800.0,9,2023-04-28 12:28:02
2,790664,849436,771812.0,5,2023-04-28 12:28:09
3,790680,849436,771828.0,13,2023-04-28 12:28:16
4,790703,849436,771851.0,5,2023-04-28 12:28:26


## accounts_userquestionrecord

In [25]:
# 데이터 불러오기 및 컬럼순서 별경
accounts_userquestionrecord = get_df('votes', 'accounts_userquestionrecord')
accounts_userquestionrecord = accounts_userquestionrecord[['id', 'user_id', 'chosen_user_id', 'question_id', 'question_piece_id', \
    'status' ,'answer_status', 'answer_updated_at', 'has_read', 'opened_times', 'report_count', 'created_at']]


# 상태컬럼, 응답상태컬럼 한글로 매핑
accounts_userquestionrecord['status'] = accounts_userquestionrecord['status'].replace({'C':'닫힘'}).replace({'I':'초성열림'}).replace({'B':'차단'})
accounts_userquestionrecord['answer_status'] = accounts_userquestionrecord['answer_status'].replace({'N':'미답변'}).replace({'P':'비공개'}).replace({'A':'공개'})


# 시간타입 변경 및 8월까지 필터링
accounts_userquestionrecord = get_datetime(accounts_userquestionrecord, 'answer_updated_at')
accounts_userquestionrecord = get_datetime(accounts_userquestionrecord, 'created_at')

accounts_userquestionrecord.head()

✅ answer_updated_at 데이트 타입 형변환 완
✅ created_at 데이트 타입 형변환 완


,id,user_id,chosen_user_id,question_id,question_piece_id,status,answer_status,answer_updated_at,has_read,opened_times,report_count,created_at
0,771777,849436,849469,252,998458,닫힘,미답변,2023-04-28 12:27:49,0,0,0,2023-04-28 12:27:49
1,771800,849436,849446,244,998459,닫힘,미답변,2023-04-28 12:28:02,0,0,0,2023-04-28 12:28:02
2,771812,849436,849454,183,998460,닫힘,미답변,2023-04-28 12:28:09,1,0,0,2023-04-28 12:28:09
3,771828,849436,847375,101,998461,닫힘,미답변,2023-04-28 12:28:16,0,0,0,2023-04-28 12:28:16
4,771851,849436,849477,209,998462,닫힘,미답변,2023-04-28 12:28:26,1,0,0,2023-04-28 12:28:26


In [ ]:
filter_point_hist, filter_question_record = \
    get_date_range(accounts_pointhistory, 'created_at', '포인트 증감 기록', accounts_userquestionrecord, 'created_at', '질문기록')

포인트 증감 기록: 2023-04-28 12:27:49 ~ 2023-08-31 16:07:56
질문기록: 2023-04-28 12:27:49 ~ 2023-08-31 16:07:56
두 시간 범위의 교집합: 2023-04-28 12:27:49 ~ 2023-08-31 16:07:56


✅ 교집합 기간으로 필터링 완료되었습니다.


In [85]:
filter_question_record = \
filter_question_record[['id', 'user_id', 'chosen_user_id', 'question_id', 'status', 'answer_status', 'answer_updated_at', 'has_read', 'report_count', 'created_at', 'opened_times', 'question_piece_id']]
filter_question_record = filter_question_record.rename(columns={'user_id':'selecting_user'})
filter_question_record.head()

,id,selecting_user,chosen_user_id,question_id,status,answer_status,answer_updated_at,has_read,report_count,created_at,opened_times,question_piece_id
0,771777,849436,849469,252,닫힘,미답변,2023-04-28 12:27:49,0,0,2023-04-28 12:27:49,0,998458
1,771800,849436,849446,244,닫힘,미답변,2023-04-28 12:28:02,0,0,2023-04-28 12:28:02,0,998459
2,771812,849436,849454,183,닫힘,미답변,2023-04-28 12:28:09,1,0,2023-04-28 12:28:09,0,998460
3,771828,849436,847375,101,닫힘,미답변,2023-04-28 12:28:16,0,0,2023-04-28 12:28:16,0,998461
4,771851,849436,849477,209,닫힘,미답변,2023-04-28 12:28:26,1,0,2023-04-28 12:28:26,0,998462


In [82]:
filter_point_hist = \
    filter_point_hist[['id', 'user_id', 'delta_point' ,'created_at', 'user_question_record_id']]
filter_point_hist.head()

,id,user_id,delta_point,created_at,user_question_record_id
0,790629,849436,9,2023-04-28 12:27:49,771777.0
1,790652,849436,9,2023-04-28 12:28:02,771800.0
2,790664,849436,5,2023-04-28 12:28:09,771812.0
3,790680,849436,13,2023-04-28 12:28:16,771828.0
4,790703,849436,5,2023-04-28 12:28:26,771851.0


### ❓ filter_point_hist에서 하나의 user_question_record_id에서 <br>+x, -300의 패턴으로 이루어지는 데이터 프레임을 이해해보자


In [105]:
df = filter_point_hist[filter_point_hist.loc[:, ['user_id', 'user_question_record_id']].duplicated(keep=False)].sort_values(by=['user_id', 'user_question_record_id', 'created_at'])
df = df.dropna()

In [106]:
df2 = filter_question_record[filter_question_record['id'].isin(df['user_question_record_id'].unique().tolist())]

In [116]:
df.head(2)

,id,user_id,delta_point,created_at,user_question_record_id
84162,3285731,838023,8,2023-05-03 11:58:02,2674185.0
84185,3286140,838023,-300,2023-05-03 11:58:27,2674185.0


In [117]:
df2.head(2)

,id,selecting_user,chosen_user_id,question_id,status,answer_status,answer_updated_at,has_read,report_count,created_at,opened_times,question_piece_id
35,772126,849438,849488,257,초성열림,미답변,2023-04-28 12:30:49,1,0,2023-04-28 12:30:49,2,998594
812,786536,850255,849995,136,초성열림,미답변,2023-04-28 14:09:54,1,0,2023-04-28 14:09:54,2,1021392


In [119]:
merge_df = pd.merge(df, df2,
         left_on='user_question_record_id',
         right_on='id')

In [ ]:
merge_df[merge_df['status'] == '초성열림']['delta_point'].value_counts() # 초성열림 200, 300, 500

delta_point
-300     38587
-200     20343
-500      6018
 15       4821
 6        4768
 7        4761
 12       4744
 9        4717
 11       4717
 13       4685
 5        4666
 14       4614
 8        4607
 10       4585
-10       4200
-1000     1349
Name: count, dtype: int64

In [128]:
merge_df[merge_df['status'] == '닫힘']['delta_point'].value_counts()

delta_point
-10     35753
 10      3340
 9       3335
 6       3299
 13      3295
 7       3277
 15      3262
 14      3235
 8       3217
 12      3206
 11      3168
 5       3167
-300        4
Name: count, dtype: int64

In [131]:
merge_df[merge_df['status'] == '차단']

,id_x,user_id,delta_point,created_at_x,user_question_record_id,id_y,selecting_user,chosen_user_id,question_id,status,answer_status,answer_updated_at,has_read,report_count,created_at_y,opened_times,question_piece_id
3320,48466495,855748,10,2023-05-11 15:10:04,24611993.0,24611993,858674,855748,398,차단,미답변,2023-05-11 15:10:04,1,0,2023-05-11 15:10:04,1,19348514
3321,48508658,855748,-300,2023-05-11 15:12:50,24611993.0,24611993,858674,855748,398,차단,미답변,2023-05-11 15:10:04,1,0,2023-05-11 15:10:04,1,19348514
3605,27995014,855832,5,2023-05-09 17:17:20,14659166.0,14659166,855424,855832,180,차단,공개,2023-05-09 17:40:05,1,0,2023-05-09 17:17:20,1,18703930
3606,28027281,855832,-300,2023-05-09 17:37:50,14659166.0,14659166,855424,855832,180,차단,공개,2023-05-09 17:40:05,1,0,2023-05-09 17:17:20,1,18703930
13618,3705699,875203,5,2023-05-03 22:55:11,2879232.0,2879232,875453,875203,111,차단,미답변,2023-05-03 22:55:11,1,0,2023-05-03 22:55:11,0,3614738
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
174021,185191422,1428542,8,2023-05-21 12:28:04,89970262.0,89970262,1428517,1428542,1281,차단,미답변,2023-05-21 12:28:04,1,0,2023-05-21 12:28:04,1,115486317
174022,185256847,1428542,-200,2023-05-21 12:31:52,89970262.0,89970262,1428517,1428542,1281,차단,미답변,2023-05-21 12:28:04,1,0,2023-05-21 12:28:04,1,115486317
188134,214707198,1481093,12,2023-05-24 06:14:05,103671612.0,103671612,1437813,1481093,426,차단,공개,2023-05-24 06:50:09,1,0,2023-05-24 06:14:05,1,131046458
188135,215058732,1481093,-200,2023-05-24 06:47:54,103671612.0,103671612,1437813,1481093,426,차단,공개,2023-05-24 06:50:09,1,0,2023-05-24 06:14:05,1,131046458
